## XAI Lab: Influence Functions and Counterfactuals

In [ ]:
pip install dice-ml "pandas<3"

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.models as models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import NearestNeighbors

import dice_ml

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Complete the coding exercises below.

#### Influence Functions:
- **Task (0.5 points):** Fine-tune the given model on the image dataset.
- **Task (1 point):** Using the helper functions, compute influence scores for a subset of training examples for a selected test image.
- **Task (1 point):** Plot the most influential training images for the chosen test image.

#### Counterfactuals (using DiCE library):
- **Task (0.5 points):** Train a logistic regression on the tabular dataset given.
- **Task (1.5 points):** Select a negatively predicted test instance and use DiCE to generate baseline counterfactual examples. Show the feature values of the original and the counterfactual.
- **Task (1.5 points):** Use DiCE's `features_to_vary` to add feasibility constraints to prevent changes to the subset of features that cannot be modified (analyse the features to determine that subset) and compare results.
- **Task (1.5 points):** Add the causal constraints given and generate counterfactuals again. Compare results.
- **Task (1.5 points):** Use DiCE's `kdtree` method to construct on-manifold counterfactuals. Compare all results.
- **Task (1 point):** Create a dot plot showing which numerical and categorical features changed for a selected test instance across the base, feasible, causal and manifold counterfactuals.

### Coding Tasks

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

class_names = train_dataset.classes

**Task (0.5 points):** Fine-tune the given model on the image dataset.

In [ ]:
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
## Your solution here here ..

In [ ]:
def get_grad_vector(model, x, y):
    model.zero_grad()
    outputs = model(x)
    loss = criterion(outputs, y)
    grads = torch.autograd.grad(loss, model.parameters())
    grad_vector = torch.cat([g.contiguous().view(-1) for g in grads])
    return grad_vector.detach()

def hvp(model, loss, params, v):
    grads = torch.autograd.grad(loss, params, create_graph=True)
    grad_vector = torch.cat([g.contiguous().view(-1) for g in grads])
    hv = torch.autograd.grad(grad_vector, params, grad_outputs=v, retain_graph=True)
    hv = torch.cat([h.contiguous().view(-1) for h in hv])
    return hv

def lissa_inverse_hvp(model, test_x, test_y, train_loader, damping=0.01, scale=25.0, recursion_depth=30):
    v = get_grad_vector(model, test_x, test_y).to(device)
    cur_estimate = v.clone()

    for i in range(recursion_depth):
        imgs, labels = next(iter(train_loader))
        imgs = imgs.to(device)
        labels = labels.to(device)

        model.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        hv = hvp(model, loss, list(model.parameters()), cur_estimate)
        cur_estimate = v + (1 - damping) * cur_estimate - hv / scale

    return cur_estimate

**Task (1 point):** Using the helper functions, compute influence scores for a subset of training examples for a selected test image.

In [ ]:
# Your solution here..

**Task (1 point):** Plot the most influential training images for the chosen test image.

In [ ]:
# Your solution here..

### Counterfactuals

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)

X = adult.data
y = adult.target

In [ ]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [ ]:
numerical_features = [
    "age",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

categorical_features = [col for col in X.columns if col not in numerical_features]

print("Numerical:", numerical_features)
print("Categorical:", categorical_features)

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])

categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

**Task (0.5 points):** Train a logistic regression on the tabular dataset given.

In [ ]:
# Your solution here..

**Task (1.5 points):** Select a negatively predicted test instance and use DiCE to generate baseline counterfactual examples. Show the feature values of the original and the counterfactual.

In [ ]:
# Your solution here..

**Task (1.5 points):** Use DiCE's `features_to_vary` to add feasibility constraints to prevent changes to the subset of features that cannot be modified (analyse the features to determine that subset) and compare results.

In [ ]:
# Your solution here..

**Task (1.5 points):** Add the causal constraints given and generate counterfactuals again. Compare results.

Causal Constraints:
age → education-num \
age → capital-gain \
education-num → hours-per-week \
hours-per-week → capital-gain

In [ ]:
# Your solution here..

**Task (1.5 points):** Use DiCE's `kdtree` method to construct on-manifold counterfactuals. Compare all results.

In [ ]:
# Your solution here..

**Task (1 point):** Create a dot plot showing which numerical and categorical features changed for a selected test instance across the base, feasible, causal and manifold counterfactuals.

In [ ]:
# Your solution here..